# 🎙️ Text-to-Speech with Emotion Control using Deep Learning
### Tacotron2-based TTS System with Emotional Speech Dataset (ESD)
---
> **v2 — `pkg_resources`-free build** &nbsp;·&nbsp; Audio I/O via `soundfile` &nbsp;·&nbsp; DSP via `scipy` &nbsp;·&nbsp; No `audioread` / No `librosa`

## Step 1: Install Dependencies
*(All audio I/O uses `soundfile` + `resampy` + `scipy` — zero `pkg_resources` dependency)*

In [1]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *args])

print('Installing packages …')

# Core
pip('torch==2.1.0', 'torchaudio==2.1.0',
    '--index-url', 'https://download.pytorch.org/whl/cpu')

# Audio I/O  (NO audioread / NO pkg_resources)
pip('soundfile==0.12.1')
pip('soxr==0.3.7')
pip('scipy==1.11.4')
pip('numpy==1.26.4')

# Viz
pip('matplotlib==3.8.2')
pip('seaborn==0.13.2')

# Data / ML
pip('pandas==2.1.4')
pip('scikit-learn==1.3.2')
pip('tqdm==4.66.1')

# Web
pip('flask==3.0.3')
pip('flask-cors==4.0.1')
pip('werkzeug==3.0.3')

# File parsing
pip('python-docx==1.1.2')
pip('PyPDF2==3.0.1')
pip('pdfminer.six==20231228')

# Text normalisation
pip('Unidecode==1.3.8')
pip('inflect==7.3.1')

# Ensure setuptools is present (fixes pkg_resources on Python 3.12)
pip('setuptools>=69.0.0')

print('\n✅ All packages installed.')

Installing packages …

✅ All packages installed.


## Step 2: Imports & Path Configuration

In [2]:
import os, sys, re, json, time, uuid, warnings, collections
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import scipy.signal       as sig
import scipy.interpolate  as interp

# Audio I/O  — pkg_resources-FREE
import soundfile as sf
import soxr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
from IPython.display import Audio, display

# ============================================================
# ⚙️  CONFIGURE YOUR PATHS HERE
# ============================================================
DATASET_ROOT = r'D:\Downloads\tts_emotion_project_v2\Emotion Speech Dataset1\English'
OUTPUT_DIR   = r'D:\7th Sem\Voxemotion\outputs'
MODEL_DIR    = r'D:\7th Sem\Voxemotion\models'
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)

EMOTIONS    = ['angry', 'happy', 'neutral', 'sad', 'surprise']
SAMPLE_RATE = 22050
HOP_LENGTH  = 256
N_MELS      = 80
N_FFT       = 1024
WIN_LENGTH  = 1024
MAX_FRAMES  = 300

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Python   : {sys.version.split()[0]}')
print(f'PyTorch  : {torch.__version__}')
print(f'NumPy    : {np.__version__}')
print(f'SoundFile: {sf.__version__}')
print(f'Device   : {device}')
print('\n✅ Imports complete — zero pkg_resources usage.')

Python   : 3.10.11
PyTorch  : 2.1.0+cu118
NumPy    : 1.26.4
SoundFile: 0.12.1
Device   : cuda

✅ Imports complete — zero pkg_resources usage.


## Step 3: Audio Utilities (soundfile + scipy only)

In [3]:
# ════════════════════════════════════════════════════════════════════
#  Complete replacement for librosa.
#  load_audio        → soundfile + resampy
#  mel spectrogram   → scipy STFT + hand-rolled mel filterbank
#  pitch shift       → scipy interpolation
#  time stretch      → scipy interpolation
# ════════════════════════════════════════════════════════════════════

def load_audio(path: str, target_sr: int = SAMPLE_RATE):
    """Read WAV → mono float32 array.  No librosa. No pkg_resources."""
    data, sr = sf.read(path, dtype='float32', always_2d=False)
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != target_sr:
        data = soxr.resample(data, sr, target_sr, quality='HQ')
    return data.astype(np.float32), target_sr


def save_audio(path: str, audio: np.ndarray, sr: int = SAMPLE_RATE):
    sf.write(path, np.clip(audio, -1.0, 1.0).astype(np.float32), sr)


def _mel_fb(sr, n_fft, n_mels):
    """Mel filterbank matrix (n_mels, n_fft//2+1)."""
    hz2mel = lambda h: 2595.0 * np.log10(1.0 + h / 700.0)
    mel2hz = lambda m: 700.0 * (10.0 ** (m / 2595.0) - 1.0)
    mels   = np.linspace(hz2mel(0), hz2mel(sr/2), n_mels + 2)
    hz_pts = np.array([mel2hz(m) for m in mels])
    bins   = np.floor((n_fft + 1) * hz_pts / sr).astype(int)
    fb     = np.zeros((n_mels, n_fft // 2 + 1), dtype=np.float32)
    for m in range(1, n_mels + 1):
        lo, c, hi = bins[m-1], bins[m], bins[m+1]
        for k in range(lo, c):
            if c != lo: fb[m-1, k] = (k-lo)/(c-lo)
        for k in range(c, hi):
            if hi != c: fb[m-1, k] = (hi-k)/(hi-c)
    return fb

_FB = _mel_fb(SAMPLE_RATE, N_FFT, N_MELS)


def compute_mel_spectrogram(audio: np.ndarray,
                             sr=SAMPLE_RATE, n_fft=N_FFT,
                             hop=HOP_LENGTH, n_mels=N_MELS):
    """Return log-mel spectrogram (n_mels, T) using scipy STFT."""
    win = np.hanning(n_fft).astype(np.float32)
    _, _, Zxx = sig.stft(audio, fs=sr, window=win,
                          nperseg=n_fft, noverlap=n_fft-hop,
                          boundary='constant', padded=True)
    power  = np.abs(Zxx) ** 2
    fb     = _FB if (n_mels==N_MELS and n_fft==N_FFT) else _mel_fb(sr,n_fft,n_mels)
    mel    = fb @ power
    mel_db = 10.0 * np.log10(np.maximum(mel, 1e-10))
    mel_db = np.maximum(mel_db, mel_db.max() - 80.0)
    return mel_db.astype(np.float32)


def pitch_shift_simple(audio: np.ndarray, sr: int, n_steps: float) -> np.ndarray:
    """Pitch shift via resampling trick — scipy only."""
    if n_steps == 0: return audio
    rate   = 2.0 ** (n_steps / 12.0)
    target = max(1, int(len(audio) / rate))
    x_old  = np.linspace(0, 1, len(audio))
    x_new  = np.linspace(0, 1, target)
    shifted = interp.interp1d(x_old, audio, 'linear', fill_value='extrapolate')(x_new)
    x_back  = np.linspace(0, 1, len(audio))
    return interp.interp1d(np.linspace(0,1,len(shifted)), shifted,
                            'linear', fill_value='extrapolate')(x_back).astype(np.float32)


def time_stretch_simple(audio: np.ndarray, rate: float) -> np.ndarray:
    """Time-stretch by linear interpolation — scipy only."""
    if rate == 1.0: return audio
    target = max(1, int(len(audio) / rate))
    return interp.interp1d(np.linspace(0,1,len(audio)), audio,
                            'linear', fill_value='extrapolate')(
           np.linspace(0,1,target)).astype(np.float32)


print('✅ Audio utilities ready.')
print('   load_audio()              — soundfile + resampy')
print('   compute_mel_spectrogram() — scipy STFT + mel filterbank')
print('   pitch_shift_simple()      — scipy interp')
print('   time_stretch_simple()     — scipy interp')

✅ Audio utilities ready.
   load_audio()              — soundfile + resampy
   compute_mel_spectrogram() — scipy STFT + mel filterbank
   pitch_shift_simple()      — scipy interp
   time_stretch_simple()     — scipy interp


## Step 4: Dataset Scanner

In [4]:
def parse_txt_file(txt_path: str) -> dict:
    mapping = {}
    try:
        with open(txt_path, 'r', encoding='utf-8', errors='replace') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    mapping[parts[0].strip()] = (
                        parts[1].strip(),
                        parts[2].strip().lower() if len(parts) > 2 else ''
                    )
    except Exception: pass
    return mapping


def scan_dataset(root: str):
    records, errors = [], []
    summary = collections.defaultdict(lambda: collections.defaultdict(int))
    root_path = Path(root)
    if not root_path.exists():
        raise FileNotFoundError(f'Not found: {root}')

    spk_dirs = sorted(d for d in root_path.iterdir() if d.is_dir())
    print(f'Found {len(spk_dirs)} speaker folder(s): {[d.name for d in spk_dirs]}\n')

    for spk in tqdm(spk_dirs, desc='Scanning'):
        txt_map = parse_txt_file(str(spk / f'{spk.name}.txt'))
        for emo_dir in sorted(d for d in spk.iterdir() if d.is_dir()):
            emotion = emo_dir.name.lower()
            for wav in sorted(emo_dir.glob('*.wav')):
                text, _ = txt_map.get(wav.stem, ('', emotion))
                readable, dur = False, 0.0
                try:
                    info = sf.info(str(wav))  # soundfile only
                    dur, readable = info.duration, True
                    summary[spk.name][emotion] += 1
                except Exception as e:
                    errors.append({'file': str(wav), 'error': str(e)})
                records.append({'speaker': spk.name, 'emotion': emotion,
                                'file': str(wav), 'filename': wav.name,
                                'text': text, 'duration': round(dur,3),
                                'readable': readable})
    return records, dict(summary), errors


print('🔍  Scanning dataset …')
records, summary, errors = scan_dataset(DATASET_ROOT)
df = pd.DataFrame(records)

total = len(df); ok = int(df['readable'].sum())
print(f'\n📊  Total: {total}  |  ✅ Readable: {ok} ({100*ok/max(total,1):.1f}%)  |  ❌ Errors: {len(errors)}')
print(f'   Total duration: {df["duration"].sum()/60:.1f} min\n')

pivot = df[df['readable']].pivot_table(index='speaker', columns='emotion',
                                        values='filename', aggfunc='count', fill_value=0)
print(pivot.to_string())
df.head()

🔍  Scanning dataset …
Found 10 speaker folder(s): ['0011', '0012', '0013', '0014', '0015', '0016', '0017', '0018', '0019', '0020']



Scanning: 100%|████████████████████████████████████████████████████████████████████████| 10/10 [04:57<00:00, 29.75s/it]


📊  Total: 17500  |  ✅ Readable: 17500 (100.0%)  |  ❌ Errors: 0
   Total duration: 804.8 min

emotion  angry  happy  neutral  sad  surprise
speaker                                      
0011       350    350      350  350       350
0012       350    350      350  350       350
0013       350    350      350  350       350
0014       350    350      350  350       350
0015       350    350      350  350       350
0016       350    350      350  350       350
0017       350    350      350  350       350
0018       350    350      350  350       350
0019       350    350      350  350       350
0020       350    350      350  350       350


,speaker,emotion,file,filename,text,duration,readable
0,0011,angry,D:\Downloads\tts_emotion_project_v2\Emotion Sp...,0011_000351.wav,"The nine the eggs, I keep.",2.349,True
1,0011,angry,D:\Downloads\tts_emotion_project_v2\Emotion Sp...,0011_000352.wav,"I did go, and made many prisoners.",2.866,True
2,0011,angry,D:\Downloads\tts_emotion_project_v2\Emotion Sp...,0011_000353.wav,That I owe my thanks to you.,2.217,True
3,0011,angry,D:\Downloads\tts_emotion_project_v2\Emotion Sp...,0011_000354.wav,They went up to the dark mass job had pointed ...,3.605,True
4,0011,angry,D:\Downloads\tts_emotion_project_v2\Emotion Sp...,0011_000355.wav,Clear than clear water!,1.887,True


## Step 5: Trial Readability Check *(No pkg_resources)*

In [5]:
def trial_readability(df: pd.DataFrame, n: int = 2) -> None:
    """
    Load and display sample audio per emotion.
    Uses soundfile + resampy only — NO librosa, NO audioread, NO pkg_resources.
    """
    print('🎧  Trial Readability Check\n')
    ok_df = df[df['readable']]
    for emotion in sorted(ok_df['emotion'].unique()):
        rows = ok_df[ok_df['emotion'] == emotion].head(n)
        for _, row in rows.iterrows():
            print(f'  ── {emotion.upper()} ─────────────────────────────')
            print(f'  File     : {row["filename"]}')
            print(f'  Text     : {row["text"] or "(no transcript)"}')
            print(f'  Duration : {row["duration"]} s')
            try:
                # ── soundfile read: no librosa, no pkg_resources ──
                audio, sr = load_audio(row['file'], target_sr=SAMPLE_RATE)
                # ─────────────────────────────────────────────────
                print(f'  Loaded   : sr={sr} Hz, samples={len(audio)}, '
                      f'min={audio.min():.4f}, max={audio.max():.4f}')
                display(Audio(audio, rate=sr))
            except Exception as e:
                print(f'  ⚠  Load error: {e}')
            print()


trial_readability(df)

🎧  Trial Readability Check

  ── ANGRY ─────────────────────────────
  File     : 0011_000351.wav
  Text     : The nine the eggs, I keep.
  Duration : 2.349 s
  Loaded   : sr=22050 Hz, samples=51795, min=-0.3109, max=0.3783



  ── ANGRY ─────────────────────────────
  File     : 0011_000352.wav
  Text     : I did go, and made many prisoners.
  Duration : 2.866 s
  Loaded   : sr=22050 Hz, samples=63195, min=-0.4819, max=0.4874



  ── HAPPY ─────────────────────────────
  File     : 0011_000701.wav
  Text     : The nine the eggs, I keep.
  Duration : 2.547 s
  Loaded   : sr=22050 Hz, samples=56168, min=-0.5076, max=0.5482



  ── HAPPY ─────────────────────────────
  File     : 0011_000702.wav
  Text     : I did go, and made many prisoners.
  Duration : 3.198 s
  Loaded   : sr=22050 Hz, samples=70516, min=-0.3674, max=0.4420



  ── NEUTRAL ─────────────────────────────
  File     : 0011_000001.wav
  Text     : The nine the eggs, I keep.
  Duration : 2.56 s
  Loaded   : sr=22050 Hz, samples=56448, min=-0.0878, max=0.1036



  ── NEUTRAL ─────────────────────────────
  File     : 0011_000002.wav
  Text     : I did go, and made many prisoners.
  Duration : 2.796 s
  Loaded   : sr=22050 Hz, samples=61652, min=-0.0978, max=0.1301



  ── SAD ─────────────────────────────
  File     : 0011_001051.wav
  Text     : The nine the eggs, I keep.
  Duration : 2.95 s
  Loaded   : sr=22050 Hz, samples=65048, min=-0.1673, max=0.1533



  ── SAD ─────────────────────────────
  File     : 0011_001052.wav
  Text     : I did go, and made many many prisoners.
  Duration : 3.42 s
  Loaded   : sr=22050 Hz, samples=75411, min=-0.1856, max=0.1840



  ── SURPRISE ─────────────────────────────
  File     : 0011_001401.wav
  Text     : The nine the eggs, I keep.
  Duration : 2.792 s
  Loaded   : sr=22050 Hz, samples=61564, min=-0.4501, max=0.4924



  ── SURPRISE ─────────────────────────────
  File     : 0011_001402.wav
  Text     : I did go, and made many prisoners.
  Duration : 3.15 s
  Loaded   : sr=22050 Hz, samples=69458, min=-0.5278, max=0.5117


## Step 6: Mel-Spectrogram Visualisation

In [6]:
def plot_mel_spectrograms(df: pd.DataFrame, save_path: str = None) -> None:
    emotions = sorted(df[df['readable']]['emotion'].unique())
    n = len(emotions)
    palette = {'angry':'#ff4040','happy':'#ffd700','neutral':'#40c8ff',
               'sad':'#7a7aff','surprise':'#ff8c00'}

    fig = plt.figure(figsize=(22, 4*n))
    fig.patch.set_facecolor('#0d0d1a')

    for i, emo in enumerate(emotions):
        row = df[(df['emotion']==emo) & df['readable']].iloc[0]
        audio, sr = load_audio(row['file'], target_sr=SAMPLE_RATE)
        mel_db    = compute_mel_spectrogram(audio)
        clr       = palette.get(emo, '#fff')

        t_ax  = np.linspace(0, len(audio)/sr, mel_db.shape[1])
        f_ax  = np.linspace(0, sr/2000, mel_db.shape[0])  # kHz

        # Mel-spec
        ax1 = fig.add_subplot(n, 2, i*2+1)
        ax1.set_facecolor('#0d0d1a')
        im = ax1.pcolormesh(t_ax, f_ax, mel_db, cmap='magma', shading='auto')
        ax1.set_title(f'Mel-Spectrogram [{emo.upper()}]', color=clr, fontsize=13, fontweight='bold')
        ax1.set_xlabel('Time (s)', color='#aaa'); ax1.set_ylabel('Freq (kHz)', color='#aaa')
        ax1.tick_params(colors='#aaa')
        [sp.set_edgecolor(clr) for sp in ax1.spines.values()]
        plt.colorbar(im, ax=ax1, format='%+.0f dB')

        # Waveform
        ax2 = fig.add_subplot(n, 2, i*2+2)
        ax2.set_facecolor('#0d0d1a')
        t_w = np.linspace(0, len(audio)/sr, len(audio))
        ax2.plot(t_w, audio, color=clr, lw=0.5, alpha=0.85)
        ax2.fill_between(t_w, audio, alpha=0.2, color=clr)
        ax2.set_title(f'Waveform [{emo.upper()}]', color=clr, fontsize=13, fontweight='bold')
        ax2.set_xlabel('Time (s)', color='#aaa'); ax2.set_ylabel('Amplitude', color='#aaa')
        ax2.tick_params(colors='#aaa'); ax2.set_xlim([0, t_w[-1]])
        [sp.set_edgecolor(clr) for sp in ax2.spines.values()]

    fig.suptitle('Mel-Spectrograms & Waveforms by Emotion',
                 color='white', fontsize=18, fontweight='bold', y=1.01)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
        print(f'Saved → {save_path}')
    plt.show(); plt.close()


plot_mel_spectrograms(df, save_path=os.path.join(OUTPUT_DIR, 'mel_spectrograms.png'))

Saved → D:\7th Sem\Voxemotion\outputs\mel_spectrograms.png


## Step 7: Feature Extraction

In [7]:
EMOTION_MAP = {e: i for i, e in enumerate(EMOTIONS)}

def extract_features(wav_path: str, max_frames: int = MAX_FRAMES) -> np.ndarray:
    audio, _ = load_audio(wav_path, target_sr=SAMPLE_RATE)
    mel = compute_mel_spectrogram(audio)
    if mel.shape[1] < max_frames:
        mel = np.pad(mel, ((0,0),(0, max_frames-mel.shape[1])), constant_values=-80.0)
    else:
        mel = mel[:, :max_frames]
    return mel.astype(np.float32)


def build_dataset(df: pd.DataFrame, max_per_class: int = 200):
    X, Y = [], []
    ok = df[df['readable'] & df['emotion'].isin(EMOTIONS)]
    for emo in EMOTIONS:
        sub = ok[ok['emotion']==emo].head(max_per_class)
        print(f'  {emo:10s}: {len(sub)} files')
        for _, row in tqdm(sub.iterrows(), total=len(sub), desc=f'  {emo}', leave=False):
            try:
                X.append(extract_features(row['file']))
                Y.append(EMOTION_MAP[emo])
            except Exception: pass
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.int64)


print('📦  Extracting features …')
X_data, y_data = build_dataset(df, max_per_class=200)
print(f'\n   X: {X_data.shape}    y: {y_data.shape}')
print(f'   Distribution: { {EMOTIONS[i]: int(v) for i,v in enumerate(np.bincount(y_data))} }')

📦  Extracting features …
  angry     : 200 files


  happy     : 200 files


  neutral   : 200 files


  sad       : 200 files


  surprise  : 200 files



   X: (1000, 80, 300)    y: (1000,)
   Distribution: {'angry': 200, 'happy': 200, 'neutral': 200, 'sad': 200, 'surprise': 200}


## Step 8: CNN-LSTM Emotion Classifier

In [8]:
class EmotionDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X).unsqueeze(1)  # (N,1,80,300)
        self.y = torch.tensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


class EmotionCNNLSTM(nn.Module):
    def __init__(self, n_cls=5, n_mels=N_MELS, frames=MAX_FRAMES):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
        )
        self.lstm = nn.LSTM(128*(n_mels//8), 256, num_layers=2,
                            batch_first=True, dropout=0.3, bidirectional=True)
        self.head = nn.Sequential(nn.Linear(512,128), nn.ReLU(), nn.Dropout(0.4), nn.Linear(128,n_cls))

    def forward(self, x):
        x = self.cnn(x)
        B,C,H,W = x.shape
        x,_ = self.lstm(x.permute(0,3,1,2).reshape(B,W,C*H))
        return self.head(x[:,-1,:])


model = EmotionCNNLSTM(len(EMOTIONS)).to(device)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Model parameters: 4,886,213


## Step 9: Training

In [9]:
from torch.optim.lr_scheduler import CosineAnnealingLR

Xtr,Xte,ytr,yte = train_test_split(X_data,y_data,test_size=0.2,random_state=42,stratify=y_data)
tr_dl = DataLoader(EmotionDS(Xtr,ytr), batch_size=32, shuffle=True)
te_dl = DataLoader(EmotionDS(Xte,yte), batch_size=32)

crit  = nn.CrossEntropyLoss()
opt   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = CosineAnnealingLR(opt, T_max=30)
hist  = {'tl':[],'vl':[],'ta':[],'va':[]}
best  = 0.0
EPOCHS = 30

print(f'Train {len(Xtr)} | Val {len(Xte)} | Device {device}\n')

for ep in range(1, EPOCHS+1):
    model.train(); tl=tc=0
    for xb,yb in tr_dl:
        xb,yb = xb.to(device),yb.to(device)
        opt.zero_grad()
        out  = model(xb); loss = crit(out,yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt.step()
        tl += loss.item()*len(yb); tc += (out.argmax(1)==yb).sum().item()
    sched.step()

    model.eval(); vl=vc=0
    with torch.no_grad():
        for xb,yb in te_dl:
            xb,yb=xb.to(device),yb.to(device)
            out=model(xb); vl+=crit(out,yb).item()*len(yb); vc+=(out.argmax(1)==yb).sum().item()

    ta=tc/len(Xtr); va=vc/len(Xte)
    for k,v in [('tl',tl/len(Xtr)),('vl',vl/len(Xte)),('ta',ta),('va',va)]: hist[k].append(v)
    if va>best:
        best=va; torch.save(model.state_dict(), os.path.join(MODEL_DIR,'emotion_best.pth'))
    if ep%5==0 or ep==1:
        print(f'Ep {ep:3d}  TrLoss {tl/len(Xtr):.4f}  TrAcc {ta:.4f}  VaLoss {vl/len(Xte):.4f}  VaAcc {va:.4f}')

print(f'\n🏆  Best Val Accuracy: {best:.4f} ({best*100:.2f}%)')

Train 800 | Val 200 | Device cuda

Ep   1  TrLoss 1.6120  TrAcc 0.1775  VaLoss 1.6122  VaAcc 0.1850
Ep   5  TrLoss 1.0116  TrAcc 0.5100  VaLoss 0.9301  VaAcc 0.5900
Ep  10  TrLoss 0.6946  TrAcc 0.7525  VaLoss 0.4368  VaAcc 0.8600
Ep  15  TrLoss 0.3317  TrAcc 0.9025  VaLoss 0.3349  VaAcc 0.9100
Ep  20  TrLoss 0.2142  TrAcc 0.9475  VaLoss 0.2415  VaAcc 0.9500
Ep  25  TrLoss 0.1000  TrAcc 0.9738  VaLoss 0.1830  VaAcc 0.9750
Ep  30  TrLoss 0.0766  TrAcc 0.9800  VaLoss 0.1902  VaAcc 0.9750

🏆  Best Val Accuracy: 0.9750 (97.50%)


## Step 10: Accuracy Report & Confusion Matrix

In [10]:
model.load_state_dict(torch.load(os.path.join(MODEL_DIR,'emotion_best.pth'), map_location=device))
model.eval()
preds,trues=[],[]
with torch.no_grad():
    for xb,yb in te_dl:
        preds.extend(model(xb.to(device)).argmax(1).cpu().tolist())
        trues.extend(yb.tolist())

print('='*55)
print('  EMOTION CLASSIFICATION ACCURACY REPORT')
print('='*55)
print(classification_report(trues,preds,target_names=EMOTIONS,digits=4))

# Curves
fig,(a1,a2)=plt.subplots(1,2,figsize=(16,5)); fig.patch.set_facecolor('#0d0d1a')
ep_r=range(1,EPOCHS+1)
for ax,(k1,k2),ttl in [(a1,('tl','vl'),'Loss'),(a2,('ta','va'),'Accuracy')]:
    ax.set_facecolor('#0d0d1a')
    ax.plot(ep_r,hist[k1],'#00e5ff',lw=2,label='Train')
    ax.plot(ep_r,hist[k2],'#ff4081',lw=2,label='Val')
    ax.set_title(f'{ttl} Curves',color='white',fontsize=14,fontweight='bold')
    ax.set_xlabel('Epoch',color='#aaa'); ax.tick_params(colors='#aaa')
    ax.legend(); ax.grid(True,alpha=0.2,color='#444')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'training_curves.png'),dpi=150,bbox_inches='tight',facecolor='#0d0d1a')
plt.show(); plt.close()

# Confusion matrix
cm=confusion_matrix(trues,preds)
fig2,ax=plt.subplots(figsize=(8,6)); fig2.patch.set_facecolor('#0d0d1a'); ax.set_facecolor('#0d0d1a')
sns.heatmap(cm,annot=True,fmt='d',cmap='YlOrRd',xticklabels=EMOTIONS,yticklabels=EMOTIONS,ax=ax)
ax.set_title('Confusion Matrix',color='white',fontsize=14,fontweight='bold'); ax.tick_params(colors='#aaa')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'confusion_matrix.png'),dpi=150,bbox_inches='tight',facecolor='#0d0d1a')
plt.show(); plt.close()

  EMOTION CLASSIFICATION ACCURACY REPORT
              precision    recall  f1-score   support

       angry     1.0000    1.0000    1.0000        40
       happy     0.9750    0.9750    0.9750        40
     neutral     0.9750    0.9750    0.9750        40
         sad     0.9737    0.9250    0.9487        40
    surprise     0.9524    1.0000    0.9756        40

    accuracy                         0.9750       200
   macro avg     0.9752    0.9750    0.9749       200
weighted avg     0.9752    0.9750    0.9749       200



## Step 11: Emotion-Aware TTS Synthesizer

In [11]:
import inflect
from unidecode import unidecode

_inflect = inflect.engine()
_SPECIAL = re.compile(r"[^a-zA-Z0-9\s.,!?'\-]")

def normalize_text(text: str) -> str:
    text = unidecode(str(text))
    text = re.sub(r'\d+', lambda m: _inflect.number_to_words(m.group()), text)
    text = _SPECIAL.sub('', text)
    return re.sub(r'\s+', ' ', text).strip()


# ── Sentence chunker: keeps each chunk under Tacotron2 decoder limit ──────────
T2_MAX_CHARS    = 150   # safe limit per chunk
SILENCE_SAMPLES = int(SAMPLE_RATE * 0.18)   # 180ms gap between sentences

def split_into_chunks(text: str, max_chars: int = T2_MAX_CHARS) -> list:
    raw = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, buf = [], ''
    for sent in raw:
        sent = sent.strip()
        if not sent: continue
        if len(buf) + len(sent) + 1 <= max_chars:
            buf = (buf + ' ' + sent).strip() if buf else sent
        else:
            if buf: chunks.append(buf)
            if len(sent) > max_chars:
                parts = re.split(r'(?<=,)\s+', sent)
                sub = ''
                for p in parts:
                    if len(sub) + len(p) + 1 <= max_chars:
                        sub = (sub + ' ' + p).strip() if sub else p
                    else:
                        if sub: chunks.append(sub)
                        while len(p) > max_chars:
                            chunks.append(p[:max_chars]); p = p[max_chars:]
                        sub = p
                if sub: chunks.append(sub)
            else:
                buf = sent
    if buf: chunks.append(buf)
    return [c for c in chunks if c.strip()]


EMOTION_PARAMS = {
    'angry'   : ( 2.0, 1.30, 1.05),
    'happy'   : ( 3.5, 1.20, 1.10),
    'neutral' : ( 0.0, 1.00, 1.00),
    'sad'     : (-3.0, 0.75, 0.88),
    'surprise': ( 4.5, 1.15, 1.15),
}


class EmotionTTS:
    WPS = 2.5   # words per second for retrieval duration estimate

    def __init__(self, df, output_dir, sr=SAMPLE_RATE):
        self.df  = df[df['readable']].copy()
        self.out = output_dir
        self.sr  = sr
        self.t2 = self.wg = self.utils = None
        self._load()

    def _load(self):
        try:
            print('⏳  Loading Tacotron2 …')
            self.t2    = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub',
                                         'nvidia_tacotron2', model_math='fp32').to(device).eval()
            self.wg    = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub',
                                         'nvidia_waveglow',  model_math='fp32').to(device).eval()
            self.utils = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_tts_utils')
            for m in self.wg.modules():
                if hasattr(m, 'weight_v'): torch.nn.utils.remove_weight_norm(m)
            print('✅  Tacotron2 + WaveGlow loaded.')
        except Exception as e:
            print(f'⚠  Tacotron2 not available ({e})')
            print('   → Retrieval-based synthesis will be used.')

    def _t2_chunk(self, chunk: str) -> np.ndarray:
        """Synthesize one short chunk via Tacotron2 (max T2_MAX_CHARS)."""
        s, l = self.utils.prepare_input_sequence([chunk])
        with torch.no_grad():
            mel, _, _ = self.t2.infer(s.to(device), l.to(device))
            audio     = self.wg.infer(mel)
        return audio[0].cpu().numpy().astype(np.float32)

    def _retrieval_chunk(self, emotion: str, dur_hint: float) -> np.ndarray:
        """Load an ESD sample trimmed to dur_hint seconds."""
        sub = self.df[self.df['emotion'] == emotion]
        if sub.empty: sub = self.df
        if sub.empty: return np.zeros(int(self.sr * dur_hint), dtype=np.float32)
        audio, _ = load_audio(sub.sample(1).iloc[0]['file'],
                               self.sr, max_seconds=dur_hint)
        needed = int(self.sr * dur_hint)
        if len(audio) < needed:
            audio = np.pad(audio, (0, needed - len(audio)))
        return audio

    def _apply_emotion(self, audio, emotion, sr):
        ps, es, ts = EMOTION_PARAMS.get(emotion, (0, 1, 1))
        if ps != 0:  audio = pitch_shift_simple(audio, sr, ps)
        audio = audio * es
        if ts != 1:  audio = time_stretch_simple(audio, ts)
        return np.clip(audio, -1.0, 1.0).astype(np.float32)

    def synthesize(self, text: str, emotion: str = 'neutral',
                   filename: str = None) -> str:
        text    = normalize_text(text)
        emotion = emotion.lower() if emotion.lower() in EMOTION_PARAMS else 'neutral'
        chunks  = split_into_chunks(text)
        if not chunks: chunks = [text[:T2_MAX_CHARS]]

        silence = np.zeros(SILENCE_SAMPLES, dtype=np.float32)
        parts   = []
        sr      = 22050 if self.t2 else self.sr

        print(f'  Synthesizing {len(chunks)} chunk(s) [{emotion}]')
        for i, chunk in enumerate(chunks):
            seg = None
            if self.t2:
                try:
                    seg = self._t2_chunk(chunk)
                except Exception as e:
                    print(f'  Chunk {i+1} Tacotron2 failed: {e}')
            if seg is None:
                dur_hint = max(2.0, len(chunk.split()) / self.WPS)
                seg = self._retrieval_chunk(emotion, dur_hint)
                sr  = self.sr
            parts.append(seg)
            if i < len(chunks) - 1:
                parts.append(silence)

        audio = np.concatenate(parts).astype(np.float32)
        audio = self._apply_emotion(audio, emotion, sr)

        fname = filename or f'tts_{emotion}_{uuid.uuid4().hex[:8]}.wav'
        path  = os.path.join(self.out, fname)
        save_audio(path, audio, sr)
        print(f'  ✅ {round(len(audio)/sr, 1)}s saved → {fname}')
        return path


tts = EmotionTTS(df, OUTPUT_DIR)
print('\n✅  TTS Synthesizer ready (sentence-chunked).')

⏳  Loading Tacotron2 …


Using cache found in C:\Users\lenovo/.cache\torch\hub\NVIDIA_DeepLearningExamples_torchhub
Using cache found in C:\Users\lenovo/.cache\torch\hub\NVIDIA_DeepLearningExamples_torchhub


✅  Tacotron2 + WaveGlow loaded.

✅  TTS Synthesizer ready (sentence-chunked).


Using cache found in C:\Users\lenovo/.cache\torch\hub\NVIDIA_DeepLearningExamples_torchhub


## Step 12: Quick Synthesis Test

In [12]:
tests = [
    ('Hello! I am so excited to meet you today.', 'happy'),
    ('I cannot believe you did that!',             'angry'),
    ('The weather today is quite pleasant.',       'neutral'),
    ('I miss those good old days.',                'sad'),
    ('Wow! I did not expect that at all!',         'surprise'),
]

for text, emotion in tests:
    print(f'\n🎙  [{emotion.upper()}]  {text}')
    out = tts.synthesize(text, emotion)
    print(f'   Saved → {out}')
    audio, sr = load_audio(out, target_sr=SAMPLE_RATE)
    display(Audio(audio, rate=sr))


🎙  [HAPPY]  Hello! I am so excited to meet you today.
  Synthesizing 1 chunk(s) [happy]
  ✅ 2.5s saved → tts_happy_7e1230cc.wav
   Saved → D:\7th Sem\Voxemotion\outputs\tts_happy_7e1230cc.wav



🎙  [ANGRY]  I cannot believe you did that!
  Synthesizing 1 chunk(s) [angry]
  ✅ 1.7s saved → tts_angry_7f684c83.wav
   Saved → D:\7th Sem\Voxemotion\outputs\tts_angry_7f684c83.wav



🎙  [NEUTRAL]  The weather today is quite pleasant.
  Synthesizing 1 chunk(s) [neutral]
  ✅ 2.3s saved → tts_neutral_fe1718c5.wav
   Saved → D:\7th Sem\Voxemotion\outputs\tts_neutral_fe1718c5.wav



🎙  [SAD]  I miss those good old days.
  Synthesizing 1 chunk(s) [sad]
  ✅ 2.5s saved → tts_sad_c966608d.wav
   Saved → D:\7th Sem\Voxemotion\outputs\tts_sad_c966608d.wav



🎙  [SURPRISE]  Wow! I did not expect that at all!
  Synthesizing 1 chunk(s) [surprise]
  ✅ 2.9s saved → tts_surprise_54d638c4.wav
   Saved → D:\7th Sem\Voxemotion\outputs\tts_surprise_54d638c4.wav


## Step 13: Save Config & Launch Web App

In [13]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path: sys.path.insert(0, PROJECT_ROOT)

with open(os.path.join(PROJECT_ROOT, 'runtime_config.json'), 'w') as f:
    json.dump({'DATASET_ROOT':DATASET_ROOT,'OUTPUT_DIR':OUTPUT_DIR,
               'MODEL_DIR':MODEL_DIR,'SAMPLE_RATE':SAMPLE_RATE,'EMOTIONS':EMOTIONS}, f, indent=2)

df.to_csv(os.path.join(OUTPUT_DIR, 'dataset_metadata.csv'), index=False)
print('Config + metadata saved.')

import subprocess as _sp, time as _t
_sp.Popen([sys.executable, os.path.join(PROJECT_ROOT, 'app.py')],
           env={**os.environ,'DATASET_ROOT':DATASET_ROOT,'OUTPUT_DIR':OUTPUT_DIR,'MODEL_DIR':MODEL_DIR})
_t.sleep(3)
print('\n🚀  Web app → http://127.0.0.1:5000')

Config + metadata saved.

🚀  Web app → http://127.0.0.1:5000
